# Fine-Tuning ngiml Model

This notebook demonstrates how to fine-tune the ngiml model on a new dataset or domain. It is adapted from the main training notebook, with configuration changes for transfer learning and fine-tuning best practices.

## 1. Clone and Prepare Repository for Fine-Tuning
Clone the ngiml repository (or update if already present) and check out the branch intended for fine-tuning.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/DeogenesMaranan/ngiml"  # update to your fork if needed
REPO_BRANCH = "main"  # or a branch specific for fine-tuning
REPO_DIR = Path("/content/ngiml")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run([
        "git",
        "clone",
        "--branch",
        REPO_BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR),
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
print(f"Repo ready at {REPO_DIR} on branch {REPO_BRANCH}")

## 2. Authenticate and Download Fine-Tuning Dataset
Authenticate with HuggingFace and download the dataset intended for fine-tuning. Use a smaller or domain-specific dataset if appropriate for transfer learning.

In [ ]:
import os
from pathlib import Path
from huggingface_hub import login, snapshot_download
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
DATASET_REPO = "your-username/your-finetune-dataset"  # update to your fine-tuning dataset
DATASET_REVISION = "main"
DATA_DIR = "/content/data_finetune"

if HF_TOKEN:
    login(token=HF_TOKEN)

os.makedirs(DATA_DIR, exist_ok=True)
snapshot_download(
    repo_id=DATASET_REPO,
    repo_type="dataset",
    local_dir=DATA_DIR,
    revision=DATASET_REVISION,
    token=HF_TOKEN,
    resume_download=True,
)

root = Path(DATA_DIR)
manifest_files = sorted(
    p for p in root.rglob("manifest.*")
    if p.name in {"manifest.parquet", "manifest.json"}
)
tar_count = sum(1 for _ in root.rglob("*.tar")) + sum(1 for _ in root.rglob("*.tar.gz")) + sum(1 for _ in root.rglob("*.tgz"))

print("Fine-tuning dataset ready at", DATA_DIR)
print("Found manifests:", [str(p) for p in manifest_files[:5]])
print("Tar shards count:", tar_count)

## 3. Mount Google Drive for Fine-Tuning Outputs
Mount Google Drive to store fine-tuning checkpoints and logs. Create a dedicated output directory for fine-tuning runs.

In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive to store fine-tuning checkpoints/logs
DRIVE_MOUNT = "/content/drive"
OUTPUT_DIR = f"{DRIVE_MOUNT}/MyDrive/ngiml_finetune_runs"

drive.mount(DRIVE_MOUNT)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Fine-tuning checkpoints will be written to", OUTPUT_DIR)

## 4. Prepare Fine-Tuning Manifest and Configurations
Load and filter the manifest for the fine-tuning dataset. Optionally, restrict to a subset of classes or samples. Save the filtered manifest for use in training.

In [ ]:
from pathlib import Path
import json
import dataclasses
from collections import Counter

from tools.manifest_utils import find_or_resolve_manifest
from tools.train_ngiml import build_default_components, build_training_config
from src.data.config import Manifest
from src.data.dataloaders import load_manifest

# Select the dataset(s) for fine-tuning
SELECTED_TRAIN_VAL_DATASETS = ["YOUR_FINETUNE_DATASET"]  # update as needed

data_root = Path(DATA_DIR)
MANIFEST_PATH = find_or_resolve_manifest(data_root)

manifest_obj = load_manifest(MANIFEST_PATH)

train_samples = [
    s for s in manifest_obj.samples
    if s.split == "train" and str(getattr(s, "dataset", "")).upper() in [d.upper() for d in SELECTED_TRAIN_VAL_DATASETS]
]
val_samples = [
    s for s in manifest_obj.samples
    if s.split == "val" and str(getattr(s, "dataset", "")).upper() in [d.upper() for d in SELECTED_TRAIN_VAL_DATASETS]
]
test_samples = [s for s in manifest_obj.samples if s.split == "test"]

filtered_samples = train_samples + val_samples + test_samples
filtered_manifest = Manifest(
    samples=filtered_samples,
    normalization_mode=manifest_obj.normalization_mode
)

FILTERED_MANIFEST_PATH = data_root / 'manifest_finetune.json'
with open(FILTERED_MANIFEST_PATH, 'w', encoding='utf-8') as handle:
    json.dump(filtered_manifest.to_dict(), handle, indent=2)
MANIFEST_PATH = FILTERED_MANIFEST_PATH

train_val_samples = train_samples + val_samples
train_val_split_counts = Counter(s.split for s in train_val_samples)
train_val_dataset_counts = Counter(s.dataset for s in train_val_samples)

print('Using filtered manifest:', MANIFEST_PATH)
print('Train/Val split counts:', dict(train_val_split_counts))
print('Train/Val dataset counts:', dict(train_val_dataset_counts))
print(f'Train samples: {len(train_samples)}, Val samples: {len(val_samples)}, Test samples: {len(test_samples)}')

## 5. Adjust Fine-Tuning Training Settings
Modify the training configuration for fine-tuning: set a lower learning rate (e.g., 1e-5 to 1e-4), reduce batch size if needed, enable resume from a pretrained checkpoint, and adjust augmentation or loss settings for fine-tuning. Optionally, freeze some layers of the model.

In [ ]:
import json
import dataclasses

from tools.colab_train_helpers import (
    apply_colab_runtime_settings,
    apply_phase2_resume_preset,
    stage_persistent_cache_to_runtime,
)

# Fine-tuning: set per-backbone and head learning rates
FINE_TUNE_LR_EFFICIENTNET = 1e-5
FINE_TUNE_LR_SWIN = 5e-6 
FINE_TUNE_LR_RESIDUAL = 2e-5
FINE_TUNE_LR_FUSION = 5e-5
FINE_TUNE_LR_DECODER = 5e-5
FINE_TUNE_BATCH_SIZE = 4
PRETRAINED_CHECKPOINT = "/content/drive/MyDrive/ngiml_runs/checkpoints/best_checkpoint.pt"

BALANCE_SAMPLING = False
PERSISTENT_CACHE_DIR = f"{OUTPUT_DIR}/local_cache"
RUNTIME_CACHE_DIR = "/content/cache"
stage_info = stage_persistent_cache_to_runtime(
    persistent_cache_dir=PERSISTENT_CACHE_DIR,
    runtime_cache_dir=RUNTIME_CACHE_DIR,
    force=False,
)
print("Cache staging:", stage_info)

model_cfg, loss_cfg, default_aug, per_dataset_aug = build_default_components()
training_config = build_training_config(
    manifest_path=MANIFEST_PATH,
    output_dir=OUTPUT_DIR,
    model_cfg=model_cfg,
    loss_cfg=loss_cfg,
    default_aug=default_aug,
    per_dataset_aug=per_dataset_aug,
)

# Apply runtime settings and fine-tuning overrides
training_config = apply_colab_runtime_settings(
    training_config,
    balance_sampling=BALANCE_SAMPLING,
    tune_for_large_batch=False,  # usually False for fine-tuning
    local_cache_dir=RUNTIME_CACHE_DIR,
)

# Set per-backbone and head learning rates for fine-tuning
if "optimizer" in training_config:
    opt_cfg = training_config["optimizer"]
    if hasattr(opt_cfg, "efficientnet"):
        opt_cfg.efficientnet.lr = FINE_TUNE_LR_EFFICIENTNET
    if hasattr(opt_cfg, "swin"):
        opt_cfg.swin.lr = FINE_TUNE_LR_SWIN
    if hasattr(opt_cfg, "residual"):
        opt_cfg.residual.lr = FINE_TUNE_LR_RESIDUAL
    if hasattr(opt_cfg, "fusion"):
        opt_cfg.fusion.lr = FINE_TUNE_LR_FUSION
    if hasattr(opt_cfg, "decoder"):
        opt_cfg.decoder.lr = FINE_TUNE_LR_DECODER

training_config["batch_size"] = FINE_TUNE_BATCH_SIZE
training_config["resume"] = PRETRAINED_CHECKPOINT
training_config["auto_resume"] = True
training_config["balance_sampling"] = BALANCE_SAMPLING
training_config["balance_real_fake"] = False
training_config["tversky_weight"] = 0.1
training_config["use_boundary_loss"] = False

print("Effective fine-tuning config:")
print(json.dumps(
    training_config,
    indent=2,
    default=lambda o: dataclasses.asdict(o) if dataclasses.is_dataclass(o) and not isinstance(o, type) else str(o),
))

## 6. Run Fine-Tuning Training Loop
Initialize the fine-tuning configuration and start the training loop using the fine-tuning settings. Ensure the model resumes from the specified pretrained checkpoint.

In [ ]:
from importlib import reload
from tools import train_ngiml

# Ensure latest module state in this kernel
reload(train_ngiml)

import importlib
try:
    import src.data.dataloaders as dl
    importlib.reload(dl)
except Exception:
    pass

# --- Fix AugmentationConfig issues before creating TrainConfig ---
from tools.train_ngiml import AugmentationConfig
if isinstance(training_config.get("default_aug"), type):
    training_config["default_aug"] = AugmentationConfig()
if isinstance(training_config.get("per_dataset_aug"), dict):
    for k, v in training_config["per_dataset_aug"].items():
        if isinstance(v, type):
            training_config["per_dataset_aug"][k] = AugmentationConfig()
# --------------------------------------------------------------

cfg = train_ngiml.TrainConfig(**training_config)
train_ngiml.run_training(cfg)

## 8. Evaluate Fine-Tuned Model on Test Split
After fine-tuning, evaluate the model on the test split using the same evaluation logic as the main notebook. Compare metrics before and after fine-tuning.

In [ ]:
# Evaluate the fine-tuned model on the test split
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

from tools.manifest_utils import find_or_resolve_manifest
from tools.infer_helpers import (
    collate_eval_batch_like_training,
    load_model_from_checkpoint,
    normalize_image_for_inference,
    should_use_high_pass_for_records,
    )
from src.data.dataloaders import load_manifest

RUN_OUTPUT_DIR = Path(training_config.get("output_dir", OUTPUT_DIR))
CHECKPOINT_DIR = RUN_OUTPUT_DIR / "checkpoints"
if not CHECKPOINT_DIR.exists():
    raise FileNotFoundError(f"Checkpoint directory not found: {CHECKPOINT_DIR}")

# Use the latest checkpoint after fine-tuning
ckpt_candidates = sorted(CHECKPOINT_DIR.glob("checkpoint_epoch_*.pt"), key=lambda p: p.stat().st_mtime)
if not ckpt_candidates:
    raise FileNotFoundError(f"No checkpoint found under {CHECKPOINT_DIR}")
CKPT_PATH = ckpt_candidates[-1]

MANIFEST_PATH = find_or_resolve_manifest(Path(DATA_DIR))
manifest = load_manifest(MANIFEST_PATH)
INFER_NORMALIZATION = str(getattr(manifest, "normalization_mode", "zero_one")).strip().lower()

model, device, ckpt_info = load_model_from_checkpoint(CKPT_PATH)
VAL_THRESHOLD = float(ckpt_info.get("default_threshold", 0.5))
INFER_MAX_SHORT_SIDE = int(ckpt_info.get("max_short_side", 0) or 0)
VAL_THRESHOLD_SOURCE = str(ckpt_info.get("threshold_source", "unknown"))

threshold_metric_name = str(training_config.get("threshold_metric", "dice")).strip().lower()
threshold_start = float(training_config.get("threshold_start", 0.1))
threshold_end = float(training_config.get("threshold_end", 0.8))
threshold_step = float(training_config.get("threshold_step", 0.02))
if threshold_step <= 0.0:
    raise ValueError(f"threshold_step must be positive, got {threshold_step}")
thresholds = np.round(np.arange(threshold_start, threshold_end + 0.5 * threshold_step, threshold_step), 6)
if thresholds.size == 0:
    raise RuntimeError("Threshold sweep grid is empty. Check threshold_start, threshold_end, and threshold_step.")

def _metrics_from_counts(tp, tn, fp, fn, eps=1e-6):
    iou = (tp + eps) / (tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    f1 = (2.0 * precision * recall) / (precision + recall + eps)
    dice = (2.0 * tp + eps) / (2.0 * tp + fp + fn + eps)
    accuracy = (tp + tn + eps) / (tp + tn + fp + fn + eps)
    return {
        "dice": float(dice),
        "iou": float(iou),
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall),
        "accuracy": float(accuracy),
    }

def _evaluate_threshold(probabilities, targets, threshold):
    tp = tn = fp = fn = 0.0
    for pred_prob, target in zip(probabilities, targets):
        pred = (pred_prob >= threshold).astype(np.float32)
        tp += float((pred * target).sum())
        tn += float(((1.0 - pred) * (1.0 - target)).sum())
        fp += float((pred * (1.0 - target)).sum())
        fn += float(((1.0 - pred) * target).sum())
    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        **_metrics_from_counts(tp, tn, fp, fn),
    }

def _sweep_thresholds(probabilities, targets):
    rows = []
    best_row = None
    best_score = None
    for threshold in thresholds:
        row = {
            "threshold": float(threshold),
            **_evaluate_threshold(probabilities, targets, float(threshold)),
        }
        metric_value = float(row.get(threshold_metric_name, row["dice"]))
        if (
            best_row is None
            or metric_value > best_score + 1e-12
            or (abs(metric_value - best_score) <= 1e-12 and row["dice"] > best_row["dice"] + 1e-12)
            or (
                abs(metric_value - best_score) <= 1e-12
                and abs(row["dice"] - best_row["dice"]) <= 1e-12
                and row["threshold"] < best_row["threshold"]
            )
        ):
            best_row = row
            best_score = metric_value
        rows.append(row)
    return rows, best_row

def _confusion_from_totals(tn, fp, fn, tp):
    return np.array([[tn, fp], [fn, tp]], dtype=np.float64)

TEST_SPLIT = "test"
EVAL_BATCH_SIZE = max(1, int(training_config.get("batch_size", 1)))
test_samples = [sample for sample in manifest.samples if sample.split == TEST_SPLIT]
if not test_samples:
    raise RuntimeError(f"No samples found for split={TEST_SPLIT!r} in manifest {MANIFEST_PATH}")

manifest_mask_paths = sum(1 for sample in test_samples if sample.mask_path is not None)
positive_labeled_samples = sum(1 for sample in test_samples if int(getattr(sample, "label", 0)) == 1)
dataset_counts = {}
for sample in test_samples:
    dataset_counts[sample.dataset] = dataset_counts.get(sample.dataset, 0) + 1

USE_HIGH_PASS_IN_EVAL = should_use_high_pass_for_records(test_samples)

print("Using checkpoint:", CKPT_PATH)
print("Using manifest:", MANIFEST_PATH)
print(f"Evaluating split: {TEST_SPLIT}")
print(f"Samples: {len(test_samples)} | manifest mask paths: {manifest_mask_paths} | positive labels: {positive_labeled_samples}")
print("Dataset counts:", dataset_counts)
print(f"Validation threshold from checkpoint: {VAL_THRESHOLD:.4f} ({VAL_THRESHOLD_SOURCE})")
print("Sweep metric:", threshold_metric_name)
print("Sweep threshold range:", [float(thresholds[0]), float(thresholds[-1]), float(threshold_step)])
print("Inference normalization:", INFER_NORMALIZATION)
print("Inference max_short_side:", INFER_MAX_SHORT_SIDE)
print("Inference high_pass enabled:", USE_HIGH_PASS_IN_EVAL)
print("Eval batch size:", EVAL_BATCH_SIZE)

# Run model inference in batches using training-parity collation (median short-side + padding).
dataset_probabilities = defaultdict(list)
dataset_targets = defaultdict(list)
all_probabilities = []
all_targets = []

model.eval()
progress = tqdm(total=len(test_samples), desc="Preparing predictions", unit="sample")
for start_idx in range(0, len(test_samples), EVAL_BATCH_SIZE):
    batch_records = test_samples[start_idx:start_idx + EVAL_BATCH_SIZE]
    batch_images, batch_masks, batch_high_pass, _ = collate_eval_batch_like_training(
        batch_records,
        max_short_side=INFER_MAX_SHORT_SIDE,
        use_high_pass=USE_HIGH_PASS_IN_EVAL,
    )

    normalized_batch_images = torch.stack(
        [normalize_image_for_inference(image, normalization_mode=INFER_NORMALIZATION) for image in batch_images],
        dim=0,
    ).to(device)

    batch_high_pass_device = None
    if batch_high_pass is not None:
        batch_high_pass_device = batch_high_pass.to(device)

    with torch.no_grad():
        logits = model(
            normalized_batch_images,
            target_size=batch_masks.shape[-2:],
            high_pass=batch_high_pass_device,
        )[-1]
        batch_probs = torch.sigmoid(logits).detach().cpu().numpy()
    batch_targets = (batch_masks[:, 0].cpu().numpy() >= 0.5).astype(np.float32)

    for local_idx, sample in enumerate(batch_records):
        pred_prob = batch_probs[local_idx, 0]
        target = batch_targets[local_idx]

        dataset_key = str(sample.dataset)
        dataset_probabilities[dataset_key].append(pred_prob)
        dataset_targets[dataset_key].append(target)
        all_probabilities.append(pred_prob)
        all_targets.append(target)

    processed = start_idx + len(batch_records)
    progress.update(len(batch_records))
progress.close()

# Threshold sweep calibration (overall + per dataset).
CALIBRATED_TEST_THRESHOLD_TABLE = {}
CALIBRATED_TEST_THRESHOLDS_BY_DATASET = {}

overall_rows, overall_best = _sweep_thresholds(all_probabilities, all_targets)
CALIBRATED_TEST_THRESHOLD = float(overall_best["threshold"])
CALIBRATED_TEST_THRESHOLD_TABLE["__overall__"] = overall_rows

for dataset_name in sorted(dataset_probabilities):
    rows, best_row = _sweep_thresholds(dataset_probabilities[dataset_name], dataset_targets[dataset_name])
    CALIBRATED_TEST_THRESHOLD_TABLE[dataset_name] = rows
    CALIBRATED_TEST_THRESHOLDS_BY_DATASET[dataset_name] = float(best_row["threshold"])

print("Calibration summary:")
print({
    "overall_best_threshold": CALIBRATED_TEST_THRESHOLD,
    "overall_best_metric": float(overall_best.get(threshold_metric_name, overall_best["dice"])),
    "per_dataset_best_thresholds": CALIBRATED_TEST_THRESHOLDS_BY_DATASET,
})

# Compare two strategies directly.
strategies = {
    "val_threshold": {
        "threshold": VAL_THRESHOLD,
        "threshold_source": "checkpoint_val_threshold",
    },
    "swept_threshold": {
        "threshold": CALIBRATED_TEST_THRESHOLD,
        "threshold_source": f"swept_test_{threshold_metric_name}",
    },
}

comparison_rows = []
per_dataset_comparison = {}
for strategy_name, info in strategies.items():
    overall_eval = _evaluate_threshold(all_probabilities, all_targets, float(info["threshold"]))
    per_dataset_eval = {}
    for dataset_name in sorted(dataset_probabilities):
        per_dataset_eval[dataset_name] = _evaluate_threshold(
            dataset_probabilities[dataset_name],
            dataset_targets[dataset_name],
            float(info["threshold"]),
        )

    per_dataset_comparison[strategy_name] = per_dataset_eval
    comparison_rows.append({
        "strategy": strategy_name,
        "threshold": float(info["threshold"]),
        "threshold_source": info["threshold_source"],
        "samples": len(all_probabilities),
        "dice": overall_eval["dice"],
        "iou": overall_eval["iou"],
        "f1": overall_eval["f1"],
        "precision": overall_eval["precision"],
        "recall": overall_eval["recall"],
        "accuracy": overall_eval["accuracy"],
        "tp": overall_eval["tp"],
        "tn": overall_eval["tn"],
        "fp": overall_eval["fp"],
        "fn": overall_eval["fn"],
    })

print("\nOverall strategy comparison:")
for row in comparison_rows:
    print({
        "strategy": row["strategy"],
        "threshold": float(row["threshold"]),
        "threshold_source": row["threshold_source"],
        "dice": float(row["dice"]),
        "iou": float(row["iou"]),
        "precision": float(row["precision"]),
        "recall": float(row["recall"]),
        "accuracy": float(row["accuracy"]),
    })

if len(comparison_rows) == 2:
    base, swept = comparison_rows
    print("\nDelta (swept_threshold - val_threshold):")
    print({
        "dice_delta": float(swept["dice"] - base["dice"]),
        "iou_delta": float(swept["iou"] - base["iou"]),
        "precision_delta": float(swept["precision"] - base["precision"]),
        "recall_delta": float(swept["recall"] - base["recall"]),
        "accuracy_delta": float(swept["accuracy"] - base["accuracy"]),
    })

print("\nPer-dataset strategy comparison:")
for dataset_name in sorted(dataset_probabilities):
    print(f"Dataset: {dataset_name}")
    for strategy_name in strategies:
        row = per_dataset_comparison[strategy_name][dataset_name]
        print({
            "strategy": strategy_name,
            "threshold": float(strategies[strategy_name]["threshold"]),
            "dice": float(row["dice"]),
            "iou": float(row["iou"]),
            "precision": float(row["precision"]),
            "recall": float(row["recall"]),
            "accuracy": float(row["accuracy"]),
        })

# Visual compare: confusion matrices for each strategy.
heatmaps = []
for row in comparison_rows:
    heatmaps.append((
        f"{row['strategy']}\nthreshold={row['threshold']:.3f}",
        _confusion_from_totals(row["tn"], row["fp"], row["fn"], row["tp"]),
    ))

fig, axes = plt.subplots(1, len(heatmaps), figsize=(6 * len(heatmaps), 4))
if len(heatmaps) == 1:
    axes = [axes]

for ax, (title, confusion_matrix) in zip(axes, heatmaps):
    im = ax.imshow(confusion_matrix, cmap="Blues")
    ax.set_title(f"{title}\nPixel Confusion Matrix")
    ax.set_xticks([0, 1], labels=["Pred Neg", "Pred Pos"])
    ax.set_yticks([0, 1], labels=["True Neg", "True Pos"])
    for row_idx in range(confusion_matrix.shape[0]):
        for col_idx in range(confusion_matrix.shape[1]):
            value = confusion_matrix[row_idx, col_idx]
            ax.text(col_idx, row_idx, f"{value:,.0f}", ha="center", va="center", color="black")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Pixel count")

plt.tight_layout()
plt.show()

print("\nSaved globals for later cells:")
print({
    "VAL_THRESHOLD": VAL_THRESHOLD,
    "VAL_THRESHOLD_SOURCE": VAL_THRESHOLD_SOURCE,
    "CALIBRATED_TEST_THRESHOLD": CALIBRATED_TEST_THRESHOLD,
    "CALIBRATED_TEST_THRESHOLDS_BY_DATASET": CALIBRATED_TEST_THRESHOLDS_BY_DATASET,
})